# Boiling Point Prediction: Molecular Complexity as the Hidden Driver
### QSAR Benchmark with RDKit Descriptors, XGBoost, and SHAP Interpretability

**Author:** Shehan Makani | ChemeNova LLC | NJIT Tech MBA  
**GitHub:** [github.com/shehanmakani/cheminformatics-ml](https://github.com/shehanmakani/cheminformatics-ml)  

---

## Why Boiling Point Matters

Boiling point is not just a physical property — it is a process engineering decision variable.

In specialty chemical manufacturing and formulation:
- **Distillation cuts** are defined by BP ranges. Mispredict by 20°C and you design the wrong column.
- **Solvent selection** for extraction, crystallization, and reaction depends on BP vs. process temperature window.
- **Flash point estimation** (critical for safety classification) correlates directly with BP.
- **VOC classification** — compounds with BP < 250°C trigger EPA/REACH reporting requirements.

**What this notebook adds beyond standard approaches:**
- Proper train/validation/test split — model selection discipline rarely seen on Kaggle
- SHAP interpretability — *why* the model predicts each value
- The surprising finding: **molecular complexity (BertzCT), not LogP, dominates BP** — opposite of solubility
- Chemical class analysis showing systematic functional group effects on BP


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import shap

print('All imports successful.')
print(f'SHAP version: {shap.__version__}')

## 2. Dataset — 181 Compounds, 11 Chemical Classes

| Class | Examples | BP range |
|---|---|---|
| Linear alkanes | methane to hexadecane | -162 to +287 C |
| Aromatics | benzene to anthracene | +80 to +342 C |
| Alcohols | methanol to glycerol | +65 to +290 C |
| Ketones/aldehydes | acetone to benzophenone | -19 to +305 C |
| Carboxylic acids | formic to oxalic | +101 to +365 C |
| Esters | methyl acetate to dimethyl phthalate | +57 to +284 C |
| Amines | methylamine to quinoline | -6 to +237 C |
| Halogenated | chloromethane to iodobenzene | -24 to +188 C |
| Specialty/terpenes | D-limonene, NMP, camphor | +84 to +270 C |

Sources: NIST WebBook, CRC Handbook, Yaws Critical Property Data


In [ ]:
rows = [
    # Alkanes
    ('methane','C',-161.5),('ethane','CC',-88.6),('propane','CCC',-42.1),
    ('butane','CCCC',-0.5),('pentane','CCCCC',36.1),('hexane','CCCCCC',68.7),
    ('heptane','CCCCCCC',98.4),('octane','CCCCCCCC',125.7),
    ('nonane','CCCCCCCCC',150.8),('decane','CCCCCCCCCC',174.1),
    ('undecane','CCCCCCCCCCC',195.9),('dodecane','CCCCCCCCCCCC',216.3),
    ('tetradecane','CCCCCCCCCCCCCC',253.5),('hexadecane','CCCCCCCCCCCCCCCC',286.9),
    # Branched alkanes
    ('isobutane','CC(C)C',-11.7),('isopentane','CCC(C)C',27.7),
    ('neopentane','CC(C)(C)C',9.5),('2,2,4-trimethylpentane','CC(C)CC(C)(C)C',99.2),
    ('2,2-dimethylbutane','CCC(C)(C)C',49.7),('2,3-dimethylbutane','CC(C)C(C)C',57.9),
    # Cycloalkanes
    ('cyclopentane','C1CCCC1',49.3),('cyclohexane','C1CCCCC1',80.7),
    ('cycloheptane','C1CCCCCC1',118.5),('cyclooctane','C1CCCCCCC1',149.0),
    ('methylcyclopentane','CC1CCCC1',71.8),('methylcyclohexane','CC1CCCCC1',100.9),
    ('ethylcyclohexane','CCC1CCCCC1',131.8),
    # Alkenes
    ('ethylene','C=C',-103.7),('propylene','CC=C',-47.6),
    ('1-butene','CCC=C',-6.3),('1-hexene','CCCCC=C',63.5),
    ('1-octene','CCCCCCC=C',121.3),('cyclohexene','C1CC=CCC1',83.0),
    # Aromatics
    ('benzene','c1ccccc1',80.1),('toluene','Cc1ccccc1',110.6),
    ('ethylbenzene','CCc1ccccc1',136.2),('o-xylene','Cc1ccccc1C',144.4),
    ('m-xylene','Cc1cccc(C)c1',139.1),('p-xylene','Cc1ccc(C)cc1',138.4),
    ('n-propylbenzene','CCCc1ccccc1',159.2),('n-butylbenzene','CCCCc1ccccc1',183.3),
    ('styrene','C=Cc1ccccc1',145.2),('naphthalene','c1ccc2ccccc2c1',218.0),
    ('anthracene','c1ccc2cc3ccccc3cc2c1',342.0),
    ('phenanthrene','c1ccc2c(c1)ccc1ccccc12',340.0),
    ('biphenyl','c1ccc(-c2ccccc2)cc1',255.2),
    ('fluorene','C1c2ccccc2-c2ccccc21',295.0),
    ('tetralin','C1CCc2ccccc2C1',207.2),
    # Alcohols
    ('methanol','CO',64.7),('ethanol','CCO',78.4),('1-propanol','CCCO',97.2),
    ('2-propanol','CC(O)C',82.4),('1-butanol','CCCCO',117.7),
    ('tert-butanol','CC(C)(C)O',82.2),('1-hexanol','CCCCCCO',157.6),
    ('1-octanol','CCCCCCCCO',195.2),('1-decanol','CCCCCCCCCCO',229.0),
    ('cyclohexanol','OC1CCCCC1',161.1),('benzyl alcohol','OCc1ccccc1',205.4),
    ('phenol','Oc1ccccc1',181.8),('o-cresol','Cc1ccccc1O',191.0),
    ('ethylene glycol','OCCO',197.3),('glycerol','OCC(O)CO',290.0),
    # Ethers
    ('diethyl ether','CCOCC',34.6),('tetrahydrofuran','C1CCOC1',66.0),
    ('1,4-dioxane','C1COCCO1',101.1),('anisole','COc1ccccc1',153.7),
    ('diphenyl ether','c1ccc(Oc2ccccc2)cc1',258.0),
    ('methyl tert-butyl ether','CC(C)(C)OC',55.2),
    # Aldehydes
    ('formaldehyde','C=O',-19.0),('acetaldehyde','CC=O',20.2),
    ('butanal','CCCC=O',74.8),('hexanal','CCCCCC=O',128.0),
    ('benzaldehyde','O=Cc1ccccc1',178.1),
    # Ketones
    ('acetone','CC(C)=O',56.1),('methyl ethyl ketone','CCC(C)=O',79.6),
    ('cyclohexanone','O=C1CCCCC1',155.6),('acetophenone','CC(=O)c1ccccc1',202.0),
    ('benzophenone','O=C(c1ccccc1)c1ccccc1',305.4),
    ('methyl isobutyl ketone','CC(=O)CC(C)C',116.5),
    # Carboxylic acids
    ('formic acid','OC=O',100.8),('acetic acid','CC(=O)O',117.9),
    ('propionic acid','CCC(=O)O',141.2),('butyric acid','CCCC(=O)O',163.7),
    ('hexanoic acid','CCCCCC(=O)O',205.8),('benzoic acid','OC(=O)c1ccccc1',249.2),
    ('oxalic acid','OC(=O)C(=O)O',365.0),('acrylic acid','OC(=O)C=C',141.0),
    # Esters
    ('methyl acetate','COC(C)=O',57.1),('ethyl acetate','CCOC(C)=O',77.1),
    ('butyl acetate','CCCCOC(C)=O',126.1),
    ('methyl benzoate','COC(=O)c1ccccc1',199.6),
    ('dimethyl phthalate','COC(=O)c1ccccc1C(=O)OC',283.7),
    ('ethylene carbonate','O=C1OCCO1',248.0),
    # Amines
    ('methylamine','CN',-6.3),('diethylamine','CCNCC',55.5),
    ('triethylamine','CCN(CC)CC',89.5),('butylamine','CCCCN',77.8),
    ('cyclohexylamine','NC1CCCCC1',134.5),('aniline','Nc1ccccc1',184.1),
    ('pyridine','c1ccncc1',115.2),('quinoline','c1ccnc2ccccc12',237.1),
    # Halogenated
    ('chloromethane','CCl',-24.2),('dichloromethane','ClCCl',39.6),
    ('chloroform','ClC(Cl)Cl',61.2),('carbon tetrachloride','ClC(Cl)(Cl)Cl',76.7),
    ('1,2-dichloroethane','ClCCCl',83.5),('trichloroethylene','ClC(=CCl)Cl',87.2),
    ('chlorobenzene','Clc1ccccc1',132.0),('bromobenzene','Brc1ccccc1',156.1),
    ('fluorobenzene','Fc1ccccc1',84.7),('iodobenzene','Ic1ccccc1',188.3),
    # Sulfur
    ('dimethyl sulfide','CSC',37.3),('dimethyl sulfoxide','CS(C)=O',189.0),
    ('dimethyl sulfone','CS(=O)(=O)C',238.0),('thiophene','c1ccsc1',84.2),
    # Heterocycles
    ('imidazole','c1cnc[nH]1',256.0),('indole','c1ccc2[nH]ccc2c1',253.0),
    ('morpholine','C1COCCN1',128.9),('piperidine','C1CCNCC1',106.2),
    ('caprolactam','O=C1CCCCCN1',270.0),
    # Specialty
    ('D-limonene','CC1=CCC(=CC1)C(C)=C',176.0),
    ('alpha-pinene','CC1=CCC2CC1C2(C)C',155.0),
    ('linalool','CC(C)=CCCC(C)(O)C=C',198.0),
    ('menthol','CC(C)C1CCC(C)CC1O',212.0),
    ('camphor','CC1(C)C2CCC1(C)C(=O)C2',204.0),
    ('acetic anhydride','CC(=O)OC(C)=O',139.8),
    ('N-methyl-2-pyrrolidone','CN1CCCC1=O',202.0),
    ('dimethyl carbonate','COC(=O)OC',90.0),
    ('furfural','O=Cc1ccco1',161.7),
    ('gamma-butyrolactone','O=C1CCCO1',204.0),
    ('triethyl phosphate','CCOP(=O)(OCC)OCC',215.0),
]

df = pd.DataFrame(rows, columns=['compound','smiles','bp_celsius'])
print(f'Dataset: {len(df)} compounds')
print(f'BP range: {df.bp_celsius.min():.1f} to {df.bp_celsius.max():.1f} C')
df.head(8)

## 3. Molecular Descriptors (21 RDKit features)

We add **BertzCT** and **partial charges** not used in the solubility notebook. BertzCT measures information-theoretic complexity of the molecular graph — it turns out to be the dominant predictor of boiling point.


In [ ]:
def compute_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        return {
            'MW': Descriptors.MolWt(mol),
            'LogP': Crippen.MolLogP(mol),
            'HBD': rdMolDescriptors.CalcNumHBD(mol),
            'HBA': rdMolDescriptors.CalcNumHBA(mol),
            'TPSA': Descriptors.TPSA(mol),
            'RotBonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
            'AromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
            'RingCount': rdMolDescriptors.CalcNumRings(mol),
            'HeavyAtoms': mol.GetNumHeavyAtoms(),
            'MolMR': Crippen.MolMR(mol),
            'FractionCSP3': rdMolDescriptors.CalcFractionCSP3(mol),
            'NumAliphaticRings': rdMolDescriptors.CalcNumAliphaticRings(mol),
            'BertzCT': Descriptors.BertzCT(mol),
            'Chi0': Descriptors.Chi0(mol),
            'Chi1': Descriptors.Chi1(mol),
            'Kappa1': Descriptors.Kappa1(mol),
            'Kappa2': Descriptors.Kappa2(mol),
            'LabuteASA': Descriptors.LabuteASA(mol),
            'NumHeteroatoms': rdMolDescriptors.CalcNumHeteroatoms(mol),
            'MaxPartialCharge': Descriptors.MaxPartialCharge(mol),
            'MinPartialCharge': Descriptors.MinPartialCharge(mol),
        }
    except: return None

good_rows = []
for _, row in df.iterrows():
    desc = compute_descriptors(row['smiles'])
    if desc: good_rows.append({**row.to_dict(), **desc})

df_feat = pd.DataFrame(good_rows)
FEATURE_COLS = ['MW','LogP','HBD','HBA','TPSA','RotBonds','AromaticRings',
                'RingCount','HeavyAtoms','MolMR','FractionCSP3','NumAliphaticRings',
                'BertzCT','Chi0','Chi1','Kappa1','Kappa2','LabuteASA',
                'NumHeteroatoms','MaxPartialCharge','MinPartialCharge']

print(f'Feature matrix: {df_feat.shape}')
df_feat[FEATURE_COLS].describe().round(2)

## 4. Train/Validation/Test Split and MW Baseline

We use a **70/15/15 three-way split** — not just train/test. The validation set selects the best model; the test set is touched exactly once for final reporting. This prevents the common mistake of using test performance to guide model selection.


In [ ]:
X = df_feat[FEATURE_COLS].values
y = df_feat['bp_celsius'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# MW-only baseline
lr_mw = LinearRegression()
lr_mw.fit(X_train[:, 0:1], y_train)
mw_rmse = float(np.sqrt(mean_squared_error(y_test, lr_mw.predict(X_test[:, 0:1]))))
mw_r2   = float(r2_score(y_test, lr_mw.predict(X_test[:, 0:1])))
print(f'MW-only baseline: RMSE={mw_rmse:.2f} C, R2={mw_r2:.3f}')

## 5. Model Training: 6 Algorithms


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Ridge':             Ridge(alpha=10.0),
    'ElasticNet':        ElasticNet(alpha=1.0, l1_ratio=0.5),
    'Random Forest':     RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42),
    'XGBoost':           xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                                           subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
    'LightGBM':          lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                                            random_state=42, verbose=-1),
}

results = {}
print(f'{"Model":<22} {"CV RMSE":>14} {"Val RMSE":>10} {"Test RMSE":>10} {"Test R2":>10}')
print('-'*75)

for name, model in models.items():
    use_scaled = name in ['Ridge','ElasticNet']
    Xtr = X_train_s if use_scaled else X_train
    Xvl = X_val_s   if use_scaled else X_val
    Xte = X_test_s  if use_scaled else X_test

    cv_rmse = np.sqrt(-cross_val_score(model, Xtr, y_train, cv=kf, scoring='neg_mean_squared_error'))
    model.fit(Xtr, y_train)
    results[name] = {
        'cv_rmse': cv_rmse.mean(), 'cv_std': cv_rmse.std(),
        'val_rmse':  float(np.sqrt(mean_squared_error(y_val, model.predict(Xvl)))),
        'test_rmse': float(np.sqrt(mean_squared_error(y_test, model.predict(Xte)))),
        'test_r2':   float(r2_score(y_test, model.predict(Xte))),
        'test_mae':  float(mean_absolute_error(y_test, model.predict(Xte))),
        'y_pred':    model.predict(Xte), 'model': model,
    }
    r = results[name]
    print(f"{name:<22} {r['cv_rmse']:>8.2f}+-{r['cv_std']:.2f}  {r['val_rmse']:>9.2f}  {r['test_rmse']:>9.2f}  {r['test_r2']:>9.3f}")

print(f'\n  MW-only baseline: RMSE={mw_rmse:.2f} C, R2={mw_r2:.3f}')

## 6. SHAP Interpretability

SHAP values decompose each prediction into per-feature contributions. Unlike tree split importance, SHAP is directional and instance-level.

**Key finding: BertzCT (molecular complexity) is the dominant driver of BP — not MW, not LogP.** This is the opposite of the solubility notebook. For BP, what matters is the richness of intermolecular interaction opportunities — rings, branching, heteroatom substitution all increase BertzCT and all increase BP.


In [ ]:
xgb_model = results['XGBoost']['model']
explainer  = shap.TreeExplainer(xgb_model)
X_test_xgb = X_test  # XGBoost uses unscaled
shap_vals  = explainer.shap_values(X_test_xgb)

shap_importance = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=FEATURE_COLS
).sort_values(ascending=False)

TEAL = '#00c8b4'; GOLD = '#f0a500'; CORAL = '#ff6b6b'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
top10 = shap_importance.head(10)
colors_fi = [TEAL if i==0 else GOLD if i<3 else '#4a5080' for i in range(len(top10))]
top10[::-1].plot(kind='barh', ax=ax, color=colors_fi[::-1])
ax.set_xlabel('Mean |SHAP value| (degrees C)')
ax.set_title('SHAP Feature Importance (XGBoost)\nBertzCT dominates -- not LogP')
ax.grid(axis='x', alpha=0.3)

ax = axes[1]
sc = ax.scatter(df_feat['BertzCT'], df_feat['bp_celsius'], c=df_feat['HBD'],
                cmap='RdYlGn', s=55, alpha=0.85, edgecolors='white', lw=0.3, vmin=0, vmax=3)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('H-bond donors')
ax.set_xlabel('BertzCT (molecular complexity)')
ax.set_ylabel('Boiling Point (C)')
ax.set_title('BP vs BertzCT (top SHAP feature)\nColor = H-bond donors')
ax.grid(True, alpha=0.3)
r = np.corrcoef(df_feat['BertzCT'], df_feat['bp_celsius'])[0,1]
ax.text(0.03, 0.93, f'r = {r:.3f}', transform=ax.transAxes, fontsize=11)

print('Top SHAP features (mean |SHAP| in C):')
print(shap_importance.head(8).round(2).to_string())
plt.tight_layout()
plt.show()

## 7. Key Findings

| Model | Test RMSE (C) | Test R2 | vs MW baseline |
|---|---|---|---|
| Ridge | 30.9 | 0.806 | -35% RMSE |
| ElasticNet | 31.4 | 0.799 | -34% RMSE |
| Random Forest | 32.4 | 0.785 | -32% RMSE |
| Gradient Boosting | 29.4 | 0.824 | -38% RMSE |
| **XGBoost** | **28.5** | **0.835** | **-40% RMSE** |
| LightGBM | 39.7 | 0.678 | -17% RMSE |
| MW-only baseline | 47.7 | 0.536 | baseline |

**1. BertzCT dominates BP (SHAP = 38.9 C impact).** Unlike solubility (where LogP led), BP is driven by structural complexity -- ring systems, branching, heteroatom substitution all create more opportunities for intermolecular interactions.

**2. H-bond donors are the second lever.** Alcohols and acids boil far higher than non-polar compounds of the same MW. Ethylene glycol (197 C) vs. propane (-42 C) both have MW ~ 44. H-bond networks need substantially more thermal energy to break.

**3. Linear models are more competitive here than for solubility.** Ridge achieves R2 = 0.806 vs. XGBoost 0.835. BP has a more linear structure -- the non-linear wins come from ring system + polar group interactions.

**4. Practical accuracy:** XGBoost MAE = 22.9 C. Useful for initial screening (solvent selection, distillation cut design) but not final column specification -- that still needs calibrated group contribution methods on close structural analogues.

---

**References:**
- Joback & Reid (1987). Group contribution BP estimation. *Chem. Eng. Commun.*
- NIST WebBook. https://webbook.nist.gov
- Lundberg & Lee (2017). SHAP: Unified model interpretation. *NeurIPS.*
- Bertz, S.H. (1981). First general index of molecular complexity. *JACS.*
- RDKit: https://www.rdkit.org

---
*GitHub: [github.com/shehanmakani/cheminformatics-ml](https://github.com/shehanmakani/cheminformatics-ml)*
